## 1. Import library dan konfigurasi eksperimen

Konfigurasi inti mengikuti modul praktikum: vocabulary maksimum 10.000 kata, embedding 100 dimensi, 64 unit untuk lapisan recurrent, Adam dengan learning rate `1e-3`, batch size 64, maksimum 100 epoch, dropout 0,3, dan early stopping dengan patience 3. Panjang urutan ditentukan dari persentil ke-95 teks latih dan dibatasi maksimum 256 token.

In [ ]:
import os
import re
import time
import random
from pathlib import Path

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Dense,
    Dropout,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    GlobalAveragePooling1D,
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Kunci seed untuk membantu reproduksibilitas eksperimen.
def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_seed()

# Konfigurasi bersama seluruh arsitektur.
VOCAB_SIZE = 10_000
MAX_LENGTH_CAP = 256
MAX_LENGTH = None  # dihitung setelah preprocessing data latih
EMBEDDING_DIM = 100
RNN_UNITS = 64
DROPOUT_RATE = 0.30
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
EPOCHS = 100
EARLY_STOPPING_PATIENCE = 3

print("TensorFlow:", tf.__version__)
print("Perangkat terdeteksi:", tf.config.list_physical_devices())
print("Seed eksperimen:", SEED)

TensorFlow: 2.21.0
Perangkat terdeteksi: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Seed eksperimen: 42


## 2. Memuat dataset

File Kaggle tidak memiliki header standar, sehingga empat kolom diberi nama `id`, `entity`, `sentiment`, dan `text`. File `twitter_training.csv` menjadi sumber untuk train/validation. File `twitter_validation.csv` dipertahankan sebagai **test set eksternal**, sesuai penggunaan file pada notebook awal.

In [ ]:
# Google Colab akan memasang Google Drive. Di Jupyter lokal, sel ini dilewati.
try:
    from google.colab import drive, files
    IS_COLAB = True
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

SEARCH_DIRS = [
    Path("/content/drive/MyDrive/Kuliah/Semester5/NLP"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
    Path("/mnt/data"),
    Path.cwd(),
]


def locate_file(filename):
    for folder in SEARCH_DIRS:
        candidate = folder / filename
        if candidate.exists() and candidate.is_file():
            return candidate
    return None

train_path = locate_file("twitter_training.csv")
test_path = locate_file("twitter_validation.csv")

if (train_path is None or test_path is None) and IS_COLAB:
    print("Unggah kedua file CSV: twitter_training.csv dan twitter_validation.csv")
    files.upload()
    train_path = locate_file("twitter_training.csv")
    test_path = locate_file("twitter_validation.csv")

if train_path is None or test_path is None:
    raise FileNotFoundError(
        "File dataset belum ditemukan. Pastikan twitter_training.csv dan "
        "twitter_validation.csv berada di folder Google Drive yang ditentukan "
        "atau diunggah ke lingkungan notebook."
    )

print("File train:", train_path)
print("File test :", test_path)


def read_twitter_csv(path):
    df = pd.read_csv(path, header=None, encoding="utf-8", engine="python")
    if df.shape[1] < 4:
        raise ValueError(f"Dataset {path} memiliki {df.shape[1]} kolom; seharusnya minimal 4 kolom.")

    # Jika ada teks berisi koma yang tidak ter-quote sehingga kolom bertambah,
    # gabungkan kembali semua kolom setelah tiga kolom pertama sebagai teks.
    if df.shape[1] > 4:
        text_column = df.iloc[:, 3:].fillna("").astype(str).agg(",".join, axis=1)
        df = pd.concat([df.iloc[:, :3].copy(), text_column.rename("text")], axis=1)

    df = df.iloc[:, :4].copy()
    df.columns = ["id", "entity", "sentiment", "text"]

    # Mengakomodasi salinan dataset yang mungkin sudah memiliki header.
    if len(df) and str(df.iloc[0]["sentiment"]).strip().lower() == "sentiment":
        df = df.iloc[1:].copy()

    df["sentiment"] = df["sentiment"].astype(str).str.strip().str.title()
    df["text"] = df["text"].fillna("").astype(str).str.strip()
    df["entity"] = df["entity"].fillna("").astype(str).str.strip()
    df = df.dropna(subset=["sentiment", "text"])
    df = df[(df["sentiment"] != "") & (df["text"] != "")]
    df = df[df["sentiment"].str.lower() != "sentiment"]
    return df.reset_index(drop=True)

train_raw = read_twitter_csv(train_path)
test_raw = read_twitter_csv(test_path)

print("Ukuran dataset training awal:", train_raw.shape)
print("Ukuran dataset test eksternal:", test_raw.shape)
print("\nDistribusi label training:")
display(train_raw["sentiment"].value_counts().rename_axis("Sentimen").to_frame("Jumlah"))
print("\nDistribusi label test:")
display(test_raw["sentiment"].value_counts().rename_axis("Sentimen").to_frame("Jumlah"))

# Pemeriksaan awal distribusi kelas pada train dan test.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_raw["sentiment"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_title("Distribusi label - training")
axes[0].set_xlabel("Sentimen")
axes[0].set_ylabel("Jumlah data")
axes[0].tick_params(axis="x", rotation=35)
test_raw["sentiment"].value_counts().sort_index().plot(kind="bar", ax=axes[1])
axes[1].set_title("Distribusi label - test eksternal")
axes[1].set_xlabel("Sentimen")
axes[1].set_ylabel("Jumlah data")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

MessageError: Error: credential propagation was unsuccessful

## 3. Preprocessing teks dan pembagian data

Preprocessing mencakup huruf kecil, penghapusan URL dan mention, pemrosesan hashtag, serta normalisasi kontraksi bahasa Inggris. Kata negasi seperti `no`, `not`, `never`, dan bentuk kontraksi yang diubah menjadi negasi dipertahankan karena dapat membalikkan makna sentimen. Stopword lain boleh dihapus, tetapi daftar negasi dikecualikan dari penghapusan.

Data dari file training dibagi secara stratified menjadi train dan validation. File `twitter_validation.csv` dipertahankan sebagai test eksternal sesuai notebook sebelumnya.

In [ ]:
try:
    nltk.download("stopwords", quiet=True)
    STOP_WORDS = set(stopwords.words("english"))
except Exception:
    # Fallback jika korpus NLTK tidak dapat diunduh.
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    STOP_WORDS = set(ENGLISH_STOP_WORDS)

NEGATION_WORDS = {"no", "nor", "not", "never", "cannot"}
CONTRACTIONS = [
    (r"\bcan't\b", "cannot"),
    (r"\bwon't\b", "will not"),
    (r"\bshan't\b", "shall not"),
    (r"n't\b", " not"),
    (r"'re\b", " are"),
    (r"'ve\b", " have"),
    (r"'ll\b", " will"),
    (r"'d\b", " would"),
    (r"'m\b", " am"),
    (r"'s\b", " is"),
]


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # URL
    text = re.sub(r"@\w+", " ", text)                   # mention
    text = re.sub(r"#(\w+)", r"\1", text)             # simpan kata hashtag
    for pattern, replacement in CONTRACTIONS:
        text = re.sub(pattern, replacement, text)
    text = re.sub(r"[^a-z\s]", " ", text)
    words = text.split()
    # Jangan menghapus negasi karena berpengaruh terhadap polaritas sentimen.
    words = [word for word in words if word not in STOP_WORDS or word in NEGATION_WORDS]
    return " ".join(words)

train_raw["clean_text"] = train_raw["text"].apply(clean_text)
test_raw["clean_text"] = test_raw["text"].apply(clean_text)
train_raw = train_raw[train_raw["clean_text"].str.strip().ne("")].reset_index(drop=True)
test_raw = test_raw[test_raw["clean_text"].str.strip().ne("")].reset_index(drop=True)

print("Contoh preprocessing:")
display(train_raw[["text", "clean_text", "sentiment"]].head(5))
print("\nContoh negasi yang dipertahankan:")
print("Input : I do not like this product!")
print("Output:", clean_text("I do not like this product!"))

# Eksplorasi panjang teks setelah preprocessing.
train_raw["clean_word_count"] = train_raw["clean_text"].str.split().str.len()
print("\nRingkasan panjang teks latih (jumlah kata):")
print(train_raw["clean_word_count"].describe().round(2))
plt.figure(figsize=(9, 4))
sns.histplot(train_raw["clean_word_count"].clip(upper=100), bins=35)
plt.title("Distribusi panjang teks latih setelah preprocessing")
plt.xlabel("Jumlah kata (nilai di atas 100 dipotong pada grafik)")
plt.ylabel("Jumlah data")
plt.tight_layout()
plt.show()

train_df, val_df = train_test_split(
    train_raw,
    test_size=0.20,
    random_state=SEED,
    stratify=train_raw["sentiment"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_raw.reset_index(drop=True)

print("\nJumlah data setelah split:")
print(f"Train      : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Test       : {len(test_df):,}")

for split_name, split_df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\nDistribusi label {split_name}:")
    display(split_df["sentiment"].value_counts().sort_index().rename_axis("Sentimen").to_frame("Jumlah"))

## 4. Label encoding, tokenisasi, padding, dan embedding

Label sentimen diubah menjadi bilangan bulat. Vocabulary dibangun hanya dari data latih untuk mencegah kebocoran informasi. Ukuran vocabulary dibatasi 10.000 kata, embedding berdimensi 100, dan panjang urutan ditetapkan pada persentil ke-95 panjang teks latih dengan batas maksimum 256 token. Padding dilakukan di bagian depan (*pre-padding*) agar kata terakhir tetap dekat dengan langkah waktu terakhir pada model recurrent.

In [ ]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["sentiment"])
known_classes = set(label_encoder.classes_)

unknown_val = sorted(set(val_df["sentiment"]) - known_classes)
unknown_test = sorted(set(test_df["sentiment"]) - known_classes)
if unknown_val or unknown_test:
    raise ValueError(
        "Ada label pada validation/test yang tidak muncul di data train. "
        f"Validation: {unknown_val}; test: {unknown_test}. Periksa konsistensi label dataset."
    )

y_val = label_encoder.transform(val_df["sentiment"])
y_test = label_encoder.transform(test_df["sentiment"])
class_names = list(label_encoder.classes_)
NUM_CLASSES = len(class_names)

print("Pemetaan label:")
for index, label in enumerate(class_names):
    print(f"{label} -> {index}")
print("Jumlah kelas:", NUM_CLASSES)

# Panjang urutan = persentil ke-95 teks latih, maksimal 256 token.
train_token_lengths = train_df["clean_text"].str.split().str.len()
MAX_LENGTH = max(1, int(min(np.percentile(train_token_lengths, 95), MAX_LENGTH_CAP)))
print(f"\nMAX_LENGTH (persentil ke-95, batas {MAX_LENGTH_CAP}):", MAX_LENGTH)

# Tokenizer hanya di-fit pada teks training.
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["clean_text"].tolist())

X_train_seq = tokenizer.texts_to_sequences(train_df["clean_text"].tolist())
X_val_seq = tokenizer.texts_to_sequences(val_df["clean_text"].tolist())
X_test_seq = tokenizer.texts_to_sequences(test_df["clean_text"].tolist())

# Pre-padding: nol ditambahkan di depan. Jika urutan terlalu panjang, bagian akhir dipotong.
X_train = pad_sequences(X_train_seq, maxlen=MAX_LENGTH, padding="pre", truncating="post")
X_val = pad_sequences(X_val_seq, maxlen=MAX_LENGTH, padding="pre", truncating="post")
X_test = pad_sequences(X_test_seq, maxlen=MAX_LENGTH, padding="pre", truncating="post")

print("\nBentuk matriks indeks:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)
print("Contoh urutan token non-padding:", X_train[0][X_train[0] != 0][:20])
print("Jumlah token unik yang tercatat tokenizer:", len(tokenizer.word_index))
print("Batas vocabulary aktif untuk model:", VOCAB_SIZE, "indeks")

non_padding_tokens = int(np.count_nonzero(X_train))
oov_tokens = int(np.count_nonzero(X_train == 1))
oov_rate = oov_tokens / max(non_padding_tokens, 1)
print(f"Proporsi token OOV di data train: {oov_rate:.2%}")

# Class weight dihitung dari data train saja untuk membantu menangani ketimpangan kelas.
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=y_train,
)
class_weight = {int(i): float(weight) for i, weight in enumerate(class_weights_array)}
print("Class weight:", {class_names[i]: round(weight, 4) for i, weight in class_weight.items()})

print("\nKonfigurasi representasi teks:")
print({
    "vocabulary_limit": VOCAB_SIZE,
    "embedding_dimension": EMBEDDING_DIM,
    "max_length": MAX_LENGTH,
    "padding": "pre",
    "truncating": "post",
})

## 5. Membangun arsitektur ANN, RNN, LSTM, GRU, dan Bidirectional

Semua arsitektur memakai embedding yang sama dari segi dimensi, lalu konfigurasi optimizer, loss, dropout, batch size, dan early stopping yang seragam. ANN menggunakan `GlobalAveragePooling1D`, sehingga tidak menyimpan urutan kata. RNN menggunakan hidden state, sedangkan LSTM dan GRU memakai gerbang untuk membantu mempertahankan informasi urutan. Varian Bidirectional dibangun setelah model gated terbaik di antara LSTM dan GRU dipilih berdasarkan macro F1 validation.

In [ ]:

def build_model(architecture, bidirectional=False):
    # Inisialisasi seed agar percobaan lebih mudah direproduksi.
    set_seed()

    model_layers = [
        Input(shape=(MAX_LENGTH,), dtype="int32"),
        Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBEDDING_DIM,
            mask_zero=True,
            name="word_embedding",
        ),
    ]

    if architecture == "ANN":
        if bidirectional:
            raise ValueError(
                "Bidirectional tidak diterapkan pada ANN."
            )

        model_layers.extend([
            GlobalAveragePooling1D(),
            Dense(64, activation="relu"),
            Dropout(DROPOUT_RATE),
        ])

    else:
        recurrent_layers = {
            "RNN": SimpleRNN,
            "LSTM": LSTM,
            "GRU": GRU,
        }

        if architecture not in recurrent_layers:
            raise ValueError(
                f"Arsitektur tidak dikenal: {architecture}"
            )

        # Perbaikan untuk mengatasi error cuDNN pada pre-padding.
        if architecture in ("LSTM", "GRU"):
            recurrent_layer = recurrent_layers[architecture](
                RNN_UNITS,
                use_cudnn=False,
            )
        else:
            recurrent_layer = recurrent_layers[architecture](
                RNN_UNITS
            )

        # Mendukung arsitektur Bidirectional.
        if bidirectional:
            recurrent_layer = Bidirectional(
                recurrent_layer,
                name="bidirectional",
            )

        model_layers.extend([
            recurrent_layer,
            Dropout(DROPOUT_RATE),
        ])

    model_layers.append(
        Dense(
            NUM_CLASSES,
            activation="softmax",
            name="sentiment_output",
        )
    )

    model_name = ("Bi" if bidirectional else "") + architecture

    model = Sequential(
        model_layers,
        name=model_name,
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


print("Konfigurasi model:")
print({
    "vocabulary": VOCAB_SIZE,
    "max_length": MAX_LENGTH,
    "embedding_dim": EMBEDDING_DIM,
    "recurrent_units": RNN_UNITS,
    "dropout": DROPOUT_RATE,
    "batch_size": BATCH_SIZE,
    "epochs_max": EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "optimizer": f"Adam({LEARNING_RATE})",
    "loss": "sparse_categorical_crossentropy",
})


## 6. Melatih empat model dasar

Model ANN, RNN, LSTM, dan GRU dilatih pada split yang sama. `EarlyStopping` memantau `val_loss`, berhenti jika tidak membaik selama tiga epoch, dan mengembalikan bobot terbaik. Class weight dihitung hanya dari label data latih.

In [ ]:

# ============================================================
# PELATIHAN MODEL ANN, RNN, LSTM, DAN GRU
# ============================================================

BASE_ARCHITECTURES = ["ANN", "RNN", "LSTM", "GRU"]

# Menyimpan model dan hasil pelatihan
models = {}
histories = {}
training_times = {}
epochs_trained = {}

for architecture in BASE_ARCHITECTURES:

    print("\n" + "=" * 60)
    print(f"Melatih model: {architecture}")
    print("=" * 60)

    # Membangun model
    model = build_model(
        architecture,
        bidirectional=False
    )

    # Mengukur waktu pelatihan
    start_time = time.perf_counter()

    # Melatih model
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=EARLY_STOPPING_PATIENCE,
                restore_best_weights=True,
                verbose=1
            )
        ],
        verbose=2
    )

    elapsed = time.perf_counter() - start_time

    # Menyimpan hasil pelatihan
    models[architecture] = model
    histories[architecture] = history.history
    training_times[architecture] = elapsed
    epochs_trained[architecture] = len(history.epoch)

    # Menampilkan ringkasan
    print(
        f"\nSelesai: {architecture}"
        f" | Waktu: {elapsed:.2f} detik"
        f" | Epoch: {len(history.epoch)}"
        f" | Parameter: {model.count_params():,}"
    )

print("\n" + "=" * 60)
print("PELATIHAN MODEL SELESAI")
print("=" * 60)

print("Model berhasil dilatih:", list(models.keys()))


## 7. Memilih model gated terbaik dan melatih satu varian Bidirectional

Sesuai modul praktikum, model yang dibandingkan untuk menentukan varian Bidirectional adalah LSTM dan GRU. Pemilihan dilakukan berdasarkan macro F1 pada validation set, bukan test set. Model terpilih kemudian dilatih dengan konfigurasi yang sama sebagai `BiLSTM` atau `BiGRU`.

In [ ]:
def metric_dict(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

validation_metrics = {}
for name, model in models.items():
    val_probs = model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
    val_pred = np.argmax(val_probs, axis=1)
    validation_metrics[name] = metric_dict(y_val, val_pred)

# Modul mengarahkan pemilihan model gated terbaik dari LSTM atau GRU.
best_gated_name = max(
    ["LSTM", "GRU"],
    key=lambda name: validation_metrics[name]["F1_macro"],
)
bi_name = "Bi" + best_gated_name
print("Skor macro F1 pada validation set:")
display(pd.DataFrame(validation_metrics).T.sort_values("F1_macro", ascending=False).round(4))
print(f"Model gated terpilih: {best_gated_name}")
print(f"Varian Bidirectional yang akan dilatih: {bi_name}")

bi_model = build_model(best_gated_name, bidirectional=True)
start_time = time.perf_counter()
bi_history = bi_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        )
    ],
    verbose=2,
)
bi_elapsed = time.perf_counter() - start_time

models[bi_name] = bi_model
histories[bi_name] = bi_history.history
training_times[bi_name] = bi_elapsed
epochs_trained[bi_name] = len(bi_history.epoch)

bi_val_pred = np.argmax(bi_model.predict(X_val, batch_size=BATCH_SIZE, verbose=0), axis=1)
validation_metrics[bi_name] = metric_dict(y_val, bi_val_pred)

print(
    f"Selesai: {bi_name} | waktu={bi_elapsed:.2f} detik | "
    f"epoch={len(bi_history.epoch)} | parameter={bi_model.count_params():,}"
)
print("\nValidasi seluruh model:")
validation_df = pd.DataFrame(validation_metrics).T.sort_values("F1_macro", ascending=False)
display(validation_df.round(4))

# Pemilihan model final berdasarkan validation set, sebelum evaluasi test.
selected_model_name = validation_df["F1_macro"].idxmax()
print("Model terpilih berdasarkan validation macro F1:", selected_model_name)

## 8. Evaluasi pada test set eksternal

Kelima model dievaluasi pada test set yang sama. Tabel memuat accuracy, precision, recall, F1 macro dan weighted, jumlah parameter, waktu pelatihan, serta epoch yang benar-benar dijalankan. Test set tidak digunakan untuk memilih arsitektur.

In [ ]:
test_predictions = {}
test_probabilities = {}
test_metric_rows = []

for name, model in models.items():
    probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    test_probabilities[name] = probabilities
    test_predictions[name] = predictions

    row = metric_dict(y_test, predictions)
    row["Parameters"] = model.count_params()
    row["Training_time_sec"] = training_times[name]
    row["Epochs_trained"] = epochs_trained[name]
    test_metric_rows.append((name, row))

comparison_df = pd.DataFrame.from_dict(dict(test_metric_rows), orient="index")
comparison_df.index.name = "Model"
comparison_df = comparison_df.sort_values("F1_macro", ascending=False)

print("Tabel perbandingan kelima model pada test set:")
display(comparison_df.round(4))

print("\nLaporan klasifikasi untuk model terpilih berdasarkan validasi:")
selected_test_pred = test_predictions[selected_model_name]
print(classification_report(
    y_test,
    selected_test_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=class_names,
    zero_division=0,
    digits=4,
))

best_test_f1_model = comparison_df["F1_macro"].idxmax()
print("Model dengan macro F1 test tertinggi (deskriptif):", best_test_f1_model)
print("Model yang dipilih sebelum melihat test set:", selected_model_name)

## 9. Kurva training vs validation: loss dan accuracy

Kurva digunakan untuk melihat apakah performa pada data latih terus membaik sementara performa validasi stagnan atau memburuk. Kondisi tersebut dapat menjadi indikasi overfitting. Karena early stopping mengembalikan bobot terbaik, jumlah epoch pada grafik tetap memperlihatkan proses pelatihan yang terjadi.

In [ ]:
model_names = list(models.keys())
fig, axes = plt.subplots(2, len(model_names), figsize=(4.5 * len(model_names), 8))
if len(model_names) == 1:
    axes = np.array(axes).reshape(2, 1)

for col, name in enumerate(model_names):
    history = histories[name]
    epochs_axis = range(1, len(history["loss"]) + 1)

    axes[0, col].plot(epochs_axis, history["loss"], label="Train loss")
    axes[0, col].plot(epochs_axis, history["val_loss"], label="Validation loss")
    axes[0, col].set_title(f"{name} - Loss")
    axes[0, col].set_xlabel("Epoch")
    axes[0, col].set_ylabel("Loss")
    axes[0, col].legend(fontsize=8)
    axes[0, col].grid(alpha=0.25)

    axes[1, col].plot(epochs_axis, history["accuracy"], label="Train accuracy")
    axes[1, col].plot(epochs_axis, history["val_accuracy"], label="Validation accuracy")
    axes[1, col].set_title(f"{name} - Accuracy")
    axes[1, col].set_xlabel("Epoch")
    axes[1, col].set_ylabel("Accuracy")
    axes[1, col].legend(fontsize=8)
    axes[1, col].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 10. Grafik perbandingan metrik test

Grafik berikut membandingkan metrik macro dan weighted agar performa pada kelas minoritas tidak hanya tertutup oleh kelas yang jumlahnya lebih besar.

In [ ]:
metric_columns = ["Accuracy", "Precision_macro", "Recall_macro", "F1_macro", "F1_weighted"]
ax = comparison_df[metric_columns].plot(kind="bar", figsize=(13, 6))
ax.set_title("Perbandingan Metrik Test: ANN, RNN, LSTM, GRU, dan Bidirectional")
ax.set_ylabel("Skor")
ax.set_xlabel("Model")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
plt.legend(title="Metrik", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 11. Confusion matrix untuk kelima model

Baris pada confusion matrix menunjukkan label aktual dan kolom menunjukkan prediksi model. Nilai diagonal adalah jumlah prediksi yang benar. Sel di luar diagonal membantu menemukan kelas sentimen yang sering tertukar.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, name in zip(axes, model_names):
    cm = confusion_matrix(y_test, test_predictions[name], labels=np.arange(NUM_CLASSES))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Label aktual")
    ax.tick_params(axis="x", rotation=35)
    ax.tick_params(axis="y", rotation=0)

for ax in axes[len(model_names):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 12. Analisis kesalahan: negasi, teks panjang, dan pengaruh Bidirectional

Analisis membandingkan model gated terpilih berdasarkan validation macro F1 dengan varian Bidirectional-nya. Akurasi diperiksa pada teks yang mengandung negasi dan teks panjang. Panjang dihitung dari jumlah token non-padding yang masuk ke model. Hasil subset harus ditafsirkan bersama jumlah contoh karena subset kecil dapat menghasilkan angka yang tidak stabil.

In [ ]:
negation_pattern = r"\b(?:no|nor|not|never|cannot|can't|don't|doesn't|didn't|isn't|aren't|wasn't|weren't|won't|wouldn't|shouldn't|couldn't)\b"
mask_negation = test_df["text"].str.contains(negation_pattern, case=False, regex=True, na=False)
sequence_lengths = np.count_nonzero(X_test, axis=1)
long_threshold = max(1, int(np.ceil(2 * MAX_LENGTH / 3)))
mask_long = sequence_lengths >= long_threshold

subset_specs = {
    "Mengandung negasi": mask_negation.to_numpy(),
    f"Teks panjang (>= {long_threshold} token)": mask_long,
}

subset_rows = []
for subset_name, mask in subset_specs.items():
    n = int(mask.sum())
    if n == 0:
        subset_rows.append({
            "Subset": subset_name,
            "Jumlah_data": 0,
            best_gated_name + "_accuracy": np.nan,
            bi_name + "_accuracy": np.nan,
            "Delta_Bidirectional": np.nan,
        })
        continue

    base_acc = accuracy_score(y_test[mask], test_predictions[best_gated_name][mask])
    bi_acc = accuracy_score(y_test[mask], test_predictions[bi_name][mask])
    subset_rows.append({
        "Subset": subset_name,
        "Jumlah_data": n,
        best_gated_name + "_accuracy": base_acc,
        bi_name + "_accuracy": bi_acc,
        "Delta_Bidirectional": bi_acc - base_acc,
    })

subset_comparison_df = pd.DataFrame(subset_rows).set_index("Subset")
print(f"Perbandingan {best_gated_name} dengan {bi_name} pada subset test:")
display(subset_comparison_df.round(4))

# Analisis akurasi berdasarkan kelompok panjang teks.
length_group = np.where(
    sequence_lengths <= max(1, MAX_LENGTH // 3),
    "Pendek",
    np.where(sequence_lengths <= max(2, (2 * MAX_LENGTH) // 3), "Sedang", "Panjang"),
)
length_rows = []
for group_name in ["Pendek", "Sedang", "Panjang"]:
    mask = length_group == group_name
    n = int(mask.sum())
    if n == 0:
        continue
    length_rows.append({
        "Kelompok_panjang": group_name,
        "Jumlah_data": n,
        best_gated_name + "_accuracy": accuracy_score(y_test[mask], test_predictions[best_gated_name][mask]),
        bi_name + "_accuracy": accuracy_score(y_test[mask], test_predictions[bi_name][mask]),
        "Delta_Bidirectional": accuracy_score(y_test[mask], test_predictions[bi_name][mask]) - accuracy_score(y_test[mask], test_predictions[best_gated_name][mask]),
    })
length_group_df = pd.DataFrame(length_rows).set_index("Kelompok_panjang")
print("\nAkurasi berdasarkan kelompok panjang teks:")
display(length_group_df.round(4))

# Contoh salah prediksi dari model terpilih berdasarkan validation macro F1.
error_df = test_df[["text", "clean_text", "sentiment"]].copy()
error_df["Prediksi"] = label_encoder.inverse_transform(test_predictions[selected_model_name])
error_df["Benar"] = error_df["sentiment"].to_numpy() == error_df["Prediksi"].to_numpy()
error_df["Mengandung_negasi"] = mask_negation.to_numpy()
error_df["Jumlah_token_model"] = sequence_lengths
error_df["Kelompok_panjang"] = length_group
misclassified_df = error_df[~error_df["Benar"]].copy()
print(f"Jumlah salah prediksi model {selected_model_name}: {len(misclassified_df):,} dari {len(error_df):,} data test")
print("Contoh maksimal 15 kesalahan prediksi pertama:")
display(misclassified_df[["text", "sentiment", "Prediksi", "Mengandung_negasi", "Kelompok_panjang"]].head(15))

## 13. Analisis overfitting, waktu, jumlah parameter, dan vanishing gradient

- **Overfitting:** indikasi awal dapat terlihat saat training loss terus lebih rendah sedangkan validation loss meningkat, atau terdapat gap besar antara training dan validation accuracy. Ini bukan bukti tunggal, sehingga kurva perlu dibaca bersama metrik test.
- **Vanishing gradient:** SimpleRNN lebih rentan karena gradien melewati banyak langkah urutan. LSTM dan GRU memakai mekanisme gate yang dirancang untuk membantu aliran informasi/gradien. Performa klasifikasi saja tidak membuktikan bahwa vanishing gradient benar-benar terjadi. Pengukuran langsung memerlukan analisis norma gradien pada langkah waktu yang berbeda.
- **Parameter dan waktu:** LSTM umumnya memiliki parameter lebih banyak daripada GRU dengan jumlah unit yang sama. Bidirectional memproses urutan dari dua arah sehingga parameternya dan waktu komputasinya biasanya meningkat. Hasil aktual tetap bergantung pada perangkat dan dataset.

In [ ]:
# Ringkasan gap train-validation pada epoch dengan validation loss terendah.
overfit_rows = []
for name, history in histories.items():
    best_epoch_idx = int(np.argmin(history["val_loss"]))
    train_acc_at_best = float(history["accuracy"][best_epoch_idx])
    val_acc_at_best = float(history["val_accuracy"][best_epoch_idx])
    gap = train_acc_at_best - val_acc_at_best
    val_loss_min = float(history["val_loss"][best_epoch_idx])
    train_loss_at_best = float(history["loss"][best_epoch_idx])
    if gap > 0.05 and val_loss_min > train_loss_at_best:
        interpretation = "Indikasi overfitting perlu diperiksa"
    elif gap > 0.03:
        interpretation = "Ada gap train-validation"
    else:
        interpretation = "Gap relatif kecil pada epoch terbaik"
    overfit_rows.append({
        "Model": name,
        "Epoch_val_loss_terbaik": best_epoch_idx + 1,
        "Train_accuracy": train_acc_at_best,
        "Validation_accuracy": val_acc_at_best,
        "Gap_accuracy": gap,
        "Train_loss": train_loss_at_best,
        "Validation_loss": val_loss_min,
        "Catatan": interpretation,
    })

overfit_df = pd.DataFrame(overfit_rows).set_index("Model")
print("Pemeriksaan indikasi overfitting pada epoch dengan validation loss terbaik:")
display(overfit_df.round(4))

print("\nPerbandingan RNN, LSTM, dan GRU:")
for name in ["RNN", "LSTM", "GRU"]:
    print(
        f"{name}: macro F1 test={comparison_df.loc[name, 'F1_macro']:.4f}; "
        f"parameter={int(comparison_df.loc[name, 'Parameters']):,}; "
        f"waktu={comparison_df.loc[name, 'Training_time_sec']:.2f} detik"
    )

print("\nEfek Bidirectional pada subset test:")
for subset_name, row in subset_comparison_df.iterrows():
    delta = row["Delta_Bidirectional"]
    if pd.isna(delta):
        print(f"{subset_name}: tidak ada data untuk dibandingkan.")
    elif delta > 0:
        print(f"{subset_name}: akurasi {bi_name} naik {delta:.4f} dibanding {best_gated_name}.")
    elif delta < 0:
        print(f"{subset_name}: akurasi {bi_name} turun {abs(delta):.4f} dibanding {best_gated_name}.")
    else:
        print(f"{subset_name}: akurasi {bi_name} sama dengan {best_gated_name}.")

print("\nKesimpulan otomatis berdasarkan data eksperimen:")
print(f"1. Model yang dipilih berdasarkan validation macro F1 adalah {selected_model_name}.")
print(f"2. Macro F1 model terpilih pada test set adalah {comparison_df.loc[selected_model_name, 'F1_macro']:.4f}.")
print(f"3. Model dengan macro F1 test tertinggi secara deskriptif adalah {best_test_f1_model}.")
print("4. Gunakan tabel metrik, kurva, confusion matrix, dan subset error untuk menjelaskan alasan pemilihan model.")
print("5. Jangan menyatakan vanishing gradient terbukti hanya dari accuracy/loss; gunakan istilah indikasi atau kerentanan arsitektur.")

## 14. Menyimpan hasil evaluasi

File CSV hasil analisis disimpan dalam folder `hasil_tugas7`. Berkas berisi tabel perbandingan test, metrik validation, pemeriksaan indikasi overfitting, perbandingan negasi/teks panjang, kelompok panjang teks, dan contoh salah prediksi. Notebook tidak menyimpan ulang dataset mentah.

In [ ]:
OUTPUT_DIR = Path("hasil_tugas7")
OUTPUT_DIR.mkdir(exist_ok=True)

comparison_df.to_csv(OUTPUT_DIR / "perbandingan_model_test.csv", index=True)
validation_df.to_csv(OUTPUT_DIR / "perbandingan_model_validation.csv", index=True)
overfit_df.to_csv(OUTPUT_DIR / "analisis_overfitting.csv", index=True)
subset_comparison_df.to_csv(OUTPUT_DIR / "analisis_negasi_teks_panjang.csv", index=True)
length_group_df.to_csv(OUTPUT_DIR / "analisis_kelompok_panjang_teks.csv", index=True)
misclassified_df.to_csv(OUTPUT_DIR / "contoh_salah_prediksi.csv", index=False)

print("File hasil tersimpan:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", path.resolve())